In [135]:
import os
import random
import sqlite3
from datetime import datetime, timedelta

import pandas as pd
from faker import Faker

In [136]:
# 1. Configuration
DB_PATH = "../data/raw/banque_entreprise.db"
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

fake = Faker('fr_FR')
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

In [137]:
# 2. Création du schéma relationnel
cursor.executescript('''
    DROP TABLE IF EXISTS factures;
    DROP TABLE IF EXISTS abonnements;
    DROP TABLE IF EXISTS clients;

    CREATE TABLE clients (
        client_id INTEGER PRIMARY KEY AUTOINCREMENT,
        nom_entreprise TEXT,
        contact_email TEXT,
        segment TEXT,
        region TEXT,
        date_inscription DATE
    );

    CREATE TABLE abonnements (
        abonnement_id INTEGER PRIMARY KEY AUTOINCREMENT,
        client_id INTEGER,
        type_forfait TEXT,
        prix_mensuel REAL,
        statut_actif BOOLEAN,
        FOREIGN KEY (client_id) REFERENCES clients (client_id)
    );

    CREATE TABLE factures (
        facture_id INTEGER PRIMARY KEY AUTOINCREMENT,
        abonnement_id INTEGER,
        montant_paye REAL,
        date_facturation DATE,
        statut_paiement TEXT,
        FOREIGN KEY (abonnement_id) REFERENCES abonnements (abonnement_id)
    );
''')

In [ ]:
# Variables de distribution métier
segments = ['PME', 'ETI', 'Grand Compte']
forfaits = {
    'Compte Pro Premium': 25.0, 
    'Location Terminal de Paiement (TPE)': 40.0, 
    'Pack Cartes Corporate Gold': 130.0, 
    'Assurance Responsabilité Civile': 210.0
}
statuts_paiement = ['Payée', 'Payée', 'Payée', 'En retard', 'Échouée'] # 60% de chances d'être payée

clients_data = []
abonnements_data = []
factures_data = []

abonnement_counter = 1

# Génération de 5000 clients
for client_id in range(1, 5001):
    date_inscription = fake.date_between(start_date='-3y', end_date='today')
    clients_data.append((
        fake.company(),
        fake.company_email(),
        random.choices(segments, weights=[0.6, 0.3, 0.1])[0],
        fake.region(),
        date_inscription.strftime('%Y-%m-%d')
    ))

    # Chaque client a entre 1 et 3 abonnements
    nb_abonnements = random.randint(1, 3)
    for _ in range(nb_abonnements):
        type_forfait = random.choice(list(forfaits.keys()))
        prix_base = forfaits[type_forfait]
        # Variation de prix (remises commerciales)
        prix_mensuel = round(prix_base * random.uniform(0.8, 1.0), 2) 
        statut_actif = random.random() > 0.15 # 85% des abonnements sont actifs

        abonnements_data.append((
            abonnement_counter,
            client_id,
            type_forfait,
            prix_mensuel,
            statut_actif
        ))

        # Génération de l'historique de facturation (6 derniers mois max)
        for mois_recul in range(6):
            date_facture = datetime.today() - timedelta(days=30 * mois_recul)
            if date_facture.date() > date_inscription:
                factures_data.append((
                    abonnement_counter,
                    prix_mensuel,
                    date_facture.strftime('%Y-%m-%d'),
                    random.choice(statuts_paiement)
                ))
        
        abonnement_counter += 1

Génération des données en cours...


In [ ]:
cursor.executemany('''
    INSERT INTO clients (nom_entreprise, contact_email, segment, region, date_inscription)
    VALUES (?, ?, ?, ?, ?)
''', clients_data)

cursor.executemany('''
    INSERT INTO abonnements (abonnement_id, client_id, type_forfait, prix_mensuel, statut_actif)
    VALUES (?, ?, ?, ?, ?)
''', abonnements_data)

cursor.executemany('''
    INSERT INTO factures (abonnement_id, montant_paye, date_facturation, statut_paiement)
    VALUES (?, ?, ?, ?)
''', factures_data)

conn.commit()
conn.close()

print(f"Base de données générée avec succès dans {DB_PATH}")
print(f"Total : {len(clients_data)} clients, {len(abonnements_data)} abonnements, {len(factures_data)} factures.")

Base de données générée avec succès dans ../data/raw/banque_entreprise.db
Total : 5000 clients, 9952 abonnements, 55390 factures.


In [140]:
query_clients = "SELECT * FROM clients"
query_factures = "SELECT * FROM factures"
query_abonnements = "SELECT * FROM abonnements"

conn = sqlite3.connect(DB_PATH)
df_clients = pd.read_sql_query(query_clients, conn)
df_factures = pd.read_sql_query(query_factures, conn)
df_abonnements = pd.read_sql_query(query_abonnements, conn)
conn.close()

In [141]:
df_clients.head()

,client_id,nom_entreprise,contact_email,segment,region,date_inscription
0,1,Texier,munozalexandrie@lopes.org,PME,Basse-Normandie,2024-06-02
1,2,Neveu S.A.S.,aimejoubert@allard.com,ETI,Centre,2026-04-17
2,3,Jourdan,josephtorres@philippe.com,Grand Compte,Mayotte,2026-02-24
3,4,Bousquet Devaux S.A.S.,corinnebourgeois@robert.com,ETI,Corse,2026-07-01
4,5,Grondin,mathildede-sousa@prevost.com,PME,Province-Alpes-Côte d'Azur,2026-05-22


In [142]:
df_factures.head()

,facture_id,abonnement_id,montant_paye,date_facturation,statut_paiement
0,1,1,23.09,2026-09-30,Échouée
1,2,1,23.09,2026-08-31,Échouée
2,3,1,23.09,2026-08-01,Payée
3,4,1,23.09,2026-07-02,Payée
4,5,1,23.09,2026-06-02,Payée


In [143]:
df_abonnements.head()

,abonnement_id,client_id,type_forfait,prix_mensuel,statut_actif
0,1,1,Compte Pro Premium,23.09,0
1,2,1,Pack Cartes Corporate Gold,126.56,1
2,3,1,Compte Pro Premium,21.66,1
3,4,2,Assurance Responsabilité Civile,177.66,1
4,5,3,Pack Cartes Corporate Gold,109.25,1
